# HyDE: Retrieve with a Hypothetical Answer Document

| Field | Value |
|---|---|
| Stage | Query transformation |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
HyDE uses generated text as a retrieval representation, never as evidence. The final answer must come from retrieved sources.

## 30-Second Summary

This notebook demonstrates HyDE with a transparent hypothetical document. A paraphrased urgency query has no lexical bridge to the incident policy; the hypothetical document supplies domain terms and retrieves it. A deliberately wrong duration shows why hypothetical facts cannot be cited.

## Why This Matters

Short questions may live far from document wording in embedding space. A plausible answer-shaped document can create a richer search representation, but it can also inject domain mismatch and hallucinated details.

## Scope

| Covers | Does not cover |
|---|---|
| Baseline vs hypothetical retrieval, evidence boundary, domain-mismatch check | Live LLM generation, neural embedding provider, production prompt benchmark |


## Mental Model

```text
question -> hypothetical document -> retrieve real sources -> discard hypothesis -> answer from evidence
```


In [1]:
import re

def tokens(text: str) -> list[str]:
    stop = {"a", "an", "and", "are", "does", "for", "how", "is", "of", "the", "to", "what", "who"}
    return [token for token in re.findall(r"[a-z0-9]+", text.lower()) if token not in stop]

def overlap_rank(query: str, documents: list[dict]) -> list[dict]:
    query_terms = set(tokens(query))
    return sorted(
        documents,
        key=lambda document: (-len(query_terms & set(tokens(document["text"]))), document["id"]),
    )

documents = [
    {"id": "access", "text": "Manager approval is required before role access is granted."},
    {"id": "billing", "text": "Billing disputes receive an invoice review within five business days."},
    {"id": "incidents", "text": "Priority one incidents have a fifteen minute acknowledgement target."},
]
query = "How quickly does urgent downtime receive confirmation?"
expected_id = "incidents"


## How It Works

HyDE asks a generator for a plausible answer-like passage, embeds or searches with that passage, and retrieves real documents. The hypothetical document can improve vocabulary alignment but has zero evidentiary authority.


## Baseline

The lexical query shares no domain term with the incident policy, so deterministic tie-breaking returns the access document.


In [2]:
baseline_top = overlap_rank(query, documents)[0]
baseline_top


{'id': 'billing',
 'text': 'Billing disputes receive an invoice review within five business days.'}

## Technique Implementation

The hypothetical document maps urgency and confirmation into the corpus vocabulary. It intentionally invents a thirty-minute value so the evidence-boundary check is visible.


In [3]:
hypothetical_document = (
    "A priority one incident caused by downtime should receive an acknowledgement "
    "within thirty minutes."
)
hyde_top = overlap_rank(hypothetical_document, documents)[0]
hypothetical_document, hyde_top


('A priority one incident caused by downtime should receive an acknowledgement within thirty minutes.',
 {'id': 'incidents',
  'text': 'Priority one incidents have a fifteen minute acknowledgement target.'})

## Controlled Experiment

We compare top-1 retrieval, then construct the final answer only from the retrieved policy. We also check whether the invented duration appears in the real evidence.


In [4]:
baseline_hit = float(baseline_top["id"] == expected_id)
hyde_hit = float(hyde_top["id"] == expected_id)
invented_value_supported = "thirty" in hyde_top["text"].lower()
final_answer = "Priority one incidents have a fifteen minute acknowledgement target [incidents]."
results = {
    "baseline_hit@1": baseline_hit,
    "hyde_hit@1": hyde_hit,
    "hypothetical_duration_supported": invented_value_supported,
    "final_answer": final_answer,
}
results


{'baseline_hit@1': 0.0,
 'hyde_hit@1': 1.0,
 'hypothetical_duration_supported': False,
 'final_answer': 'Priority one incidents have a fifteen minute acknowledgement target [incidents].'}

## Evaluation

The original lexical query misses; the hypothetical document retrieves `incidents`. Its invented **thirty-minute** detail conflicts with the real **fifteen-minute** policy, so the final answer discards the hypothesis and cites only retrieved evidence.


In [5]:
assert results["baseline_hit@1"] == 0.0
assert results["hyde_hit@1"] == 1.0
assert not results["hypothetical_duration_supported"]
assert "fifteen" in results["final_answer"] and "thirty" not in results["final_answer"]
print("HyDE evidence-boundary checks passed.")


HyDE evidence-boundary checks passed.


## Decision Guide

| Situation | Choice |
|---|---|
| Short abstract query, weak dense recall | Evaluate HyDE |
| Exact IDs/names | Sparse/direct retrieval |
| Strict latency/cost | Avoid extra generation unless gain is proven |
| High hallucination consequence | Keep hypothesis isolated and require cited evidence |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Wrong domain retrieved | Hypothesis follows model prior | Domain prompt/router and baseline fusion |
| Hypothetical fact appears in answer | Evidence boundary violated | Discard hypothesis before generation |
| No gain | Query already matches corpus | Route around HyDE |
| Latency doubles | Extra generation step | Cache, smaller model, timeout, measure by segment |


## Production Notes

### Observability
Log hypothesis hash/text under approved policy, retrieved IDs, baseline-vs-HyDE delta, model/version, latency, and fallback.

### Safety and Guardrails
Treat hypothesis as untrusted generated text and never cite it.

### Latency and Cost
HyDE adds a generation call before retrieval; use only for segments with measured recall gain.


## Practice

Create a hypothesis for an out-of-domain query and verify that a minimum relevance threshold causes abstention rather than forced retrieval.

## Recall

Toggle - Recall: Is the hypothetical document evidence?
No. It is only a retrieval representation.

Toggle - Recall: What proves HyDE is useful?
A controlled recall/quality gain that justifies the extra generation cost.

## Sources

- [Precise Zero-Shot Dense Retrieval without Relevance Labels (HyDE)](https://arxiv.org/abs/2212.10496)
- Repository-owned synthetic policy fixture in this notebook

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the evidence-boundary demonstration | Evaluate real embeddings and domain-routed hypotheses |
